# RNFL v4 (KAGGLE): embedding beku + FITUR RADIAL berbasis domain (striasi RNFL) + nested CV per pasien + ensemble + stacking tingkat mata
Mengapa pendekatan ini: dengan 127 pasien, fine-tuning CNN penuh mudah overfit (train 0,91 vs validasi 0,68).
Di sini backbone ImageNet dibekukan, fitur diekstrak SEKALI (termasuk layer tengah yang lebih peka terhadap tekstur),
lalu classifier linear dengan PCA + regularisasi dipilih lewat nested CV (inner split per pasien). Iterasi sangat cepat, hasil mudah direproduksi.
Tambahan v4: fitur tekstur berorientasi radial (peka striasi serat) -> pada uji saya sebanding/lebih baik daripada embedding ImageNet.
Tidak ada jaminan mencapai target; sel laporan melaporkan apakah target tercapai pada validasi per pasien.

In [ ]:
import os, glob, re, time, shutil, zipfile, numpy as np, pandas as pd, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.metrics import (accuracy_score, f1_score, balanced_accuracy_score, roc_auc_score,
                             confusion_matrix, recall_score)
import matplotlib.pyplot as plt

START_TIME = time.time()
print(tf.__version__, tf.config.list_physical_devices('GPU'))   # GPU hanya dipakai untuk ekstraksi embedding

INPUT_ROOT = '/kaggle/input'
CSV_NAME   = 'label_sector_clean.csv'
ZIP_NAME   = 'sector_clean.zip'
OUT_DIR    = '/kaggle/working/hasil_training/'
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# ---- Lokasi data (otomatis di /kaggle/input) ----
csv_found = glob.glob(f'{INPUT_ROOT}/**/{CSV_NAME}', recursive=True)
if not csv_found:
    print('Isi', INPUT_ROOT, ':'); print(*sorted(glob.glob(f'{INPUT_ROOT}/*') + glob.glob(f'{INPUT_ROOT}/*/*'))[:30], sep='\n')
    raise FileNotFoundError(f'{CSV_NAME} tidak ditemukan. Pastikan dataset sudah ditambahkan lewat Add Input.')
CSV_PATH = csv_found[0]

img_found = glob.glob(f'{INPUT_ROOT}/**/*_sector_0.png', recursive=True)
if img_found:                                    # Kaggle sudah mengekstrak zip menjadi folder gambar
    IMAGE_DIR = os.path.dirname(img_found[0]) + '/'
else:                                            # zip belum terekstrak -> ekstrak ke disk sementara
    zp = glob.glob(f'{INPUT_ROOT}/**/{ZIP_NAME}', recursive=True)
    if not zp: raise FileNotFoundError(f'Gambar maupun {ZIP_NAME} tidak ditemukan di {INPUT_ROOT}')
    with zipfile.ZipFile(zp[0]) as z: z.extractall('/tmp/rnfl_img/')
    IMAGE_DIR = os.path.dirname(glob.glob('/tmp/rnfl_img/**/*_sector_0.png', recursive=True)[0]) + '/'

# Lanjutkan dari output versi sebelumnya bila dilampirkan sebagai Input (Add Input > Notebook Output Files)
for p in glob.glob(f'{INPUT_ROOT}/**/emb_*.npy', recursive=True):      # pakai ulang embedding dari output versi sebelumnya (hemat waktu GPU)
    if 'hasil_training' in p and not os.path.exists(OUT_DIR + os.path.basename(p)):
        shutil.copy(p, OUT_DIR)

print('CSV   :', CSV_PATH)
print('Gambar:', IMAGE_DIR, '->', len(os.listdir(IMAGE_DIR)), 'file (harus 5180)')

In [ ]:
df_raw = pd.read_csv(CSV_PATH)
print(df_raw.shape); display(df_raw.head()); print(df_raw['label'].value_counts().sort_index())

In [ ]:
# ================= KONFIGURASI =================
FILE_COL, LABEL_COL = 'file_name', 'label'
GROUP_REGEX = r'^(fff_\d+_\d+)_'
NAME_MAP = {1: 'kelas_1', 2: 'kelas_2', 3: 'kelas_3', 4: 'kelas_4'}

TASK = 'binary'                 # 'binary' | 'multiclass4'
BINARY_NORMAL_CLASSES = [1]     # ASUMSI kelas 1 = normal (konfirmasi ke pembimbing)
EYE_ABNORMAL_MIN = 3            # mata abnormal bila >= K sektor abnormal (definisi klinis: tentukan bersama dokter)
TARGET = 0.88                   # hanya untuk laporan akhir; TIDAK dipakai untuk memilih model/hyperparameter

EMB_SIZE   = (224, 224)         # ukuran input ekstraksi embedding
BACKBONES  = ['ResNet50', 'MobileNetV2', 'InceptionV3']
N_FOLDS, SEED = 5, 42
PARAM_GRID = {'ct__emb__pca__n_components': [32, 64, 128],       # dipilih lewat INNER CV per pasien (bukan di fold evaluasi)
              'lr__C': [0.003, 0.01, 0.03, 0.1, 0.3]}
CLASS_WEIGHT = 'balanced'

RUN_TAG   = f'{TASK}_emb{EMB_SIZE[0]}'
SCENARIOS = [('emb', 0.0)]
USE_RADIAL = True                 # gabungkan fitur radial ke setiap backbone (model 'RADIAL' = fitur radial saja)
ARCHS     = BACKBONES + ['RADIAL']

In [ ]:
df = df_raw.copy()
df['path'] = df[FILE_COL].astype(str).apply(lambda f: os.path.join(IMAGE_DIR, f))
missing = ~df['path'].apply(os.path.exists)
print('File hilang:', missing.sum(), 'dari', len(df))
df = df[~missing].reset_index(drop=True)

df['group'] = df[FILE_COL].astype(str).str.extract(GROUP_REGEX)[0]
assert df['group'].notna().all(), 'ada nama file yang tidak cocok GROUP_REGEX'
print('Jumlah pasien:', df['group'].nunique(), '| baris:', len(df))

# Sisi mata: l = kiri; r dan u = kanan
df['side'] = df[FILE_COL].astype(str).str.extract(r'^fff_\d+_\d+_([a-z])_')[0].map({'l': 'L', 'r': 'R', 'u': 'R'})
assert df['side'].notna().all(), 'ada sisi mata tak dikenal'
df['sec'] = df[FILE_COL].astype(str).str.extract(r'sector_(\d+)')[0].astype(int)
# Anatomis mata kiri = bayangan cermin mata kanan: sektor k (kiri) ~ sektor (10-k)%12 (kanan)
df['csec'] = np.where(df['side'] == 'L', (10 - df['sec']) % 12, df['sec'])
df['flip'] = (df['side'] == 'L').astype(int)
df['unit'] = df[FILE_COL].astype(str).str.extract(r'^(.*)_sector_')[0]      # 1 citra (1 scan 1 mata) = sampai 12 sektor
df['eye']  = df['group'] + '_' + df['side'].str.lower()                     # 1 mata fisik (gabungan scan ulang)
df['abn']  = (~df[LABEL_COL].isin(BINARY_NORMAL_CLASSES)).astype(int)

if TASK == 'binary':
    df['y'] = df['abn']; CLASS_NAMES = ['normal', 'abnormal']
else:
    le = LabelEncoder(); df['y'] = le.fit_transform(df[LABEL_COL])
    CLASS_NAMES = [NAME_MAP.get(c, str(c)) for c in le.classes_]
NUM_CLASSES = len(CLASS_NAMES)
print(CLASS_NAMES); print(df['y'].value_counts().sort_index())
print('Pasien per kelas:', {n: df.loc[df['y'] == i, 'group'].nunique() for i, n in enumerate(CLASS_NAMES)})
_u = df.groupby('unit')['abn'].sum()
print(f'Tingkat mata: {df["eye"].nunique()} mata, {len(_u)} citra; abnormal (>= {EYE_ABNORMAL_MIN} sektor abnormal): {(_u >= EYE_ABNORMAL_MIN).mean():.1%} citra')

In [ ]:
# ================= FITUR RADIAL berbasis domain (CPU; sekali, disimpan) =================
from joblib import Parallel, delayed
def radial_feats(path, flip, n_out=52):
    """Fitur tekstur berorientasi radial (peka striasi serat RNFL). Wedge dicrop; arah radial dihitung dari pusat kanvas (ONH)."""
    import numpy as np
    from PIL import Image
    from scipy import ndimage as ndi
    a = np.asarray(Image.open(path).convert('RGB')).astype(np.float32)
    if flip: a = a[:, ::-1, :]
    m = a.max(2) > 12; ys, xs = np.where(m)
    if len(ys) < 30: return np.zeros(n_out)
    y0, y1, x0, x1 = max(ys.min() - 10, 0), ys.max() + 11, max(xs.min() - 10, 0), xs.max() + 11
    a = a[y0:y1, x0:x1]; m = m[y0:y1, x0:x1]
    H, W = m.shape; yy, xx = np.mgrid[0:H, 0:W]
    phi = np.arctan2((yy + y0) - 224.0, (xx + x0) - 224.0)          # arah radial dari pusat kanvas
    me = ndi.binary_erosion(m, iterations=4)
    if me.sum() < 20: me = m
    g, r, b = a[:, :, 1], a[:, :, 0], a[:, :, 2]
    out = []
    for ch in (g, r):
        v = ch[m]; out += [v.mean(), v.std(), *np.percentile(v, [10, 50, 90])]
    out += [r[m].mean() / (g[m].mean() + 1e-6), b[m].mean() / (g[m].mean() + 1e-6)]
    mu = ndi.gaussian_filter(g * m, 6) / (ndi.gaussian_filter(m.astype(np.float32), 6) + 1e-6)
    gn = (g - mu) * m                                                # normalisasi iluminasi lokal
    for s in (1.0, 2.0, 3.0):
        gx = ndi.gaussian_filter(gn, s, order=(0, 1)); gy = ndi.gaussian_filter(gn, s, order=(1, 0))
        mag = np.hypot(gx, gy)[me]; th = np.arctan2(gy, gx)[me]; ph = phi[me]
        d = np.mod(th - ph, np.pi)                                   # orientasi gradien relatif terhadap arah radial
        h, _ = np.histogram(d, bins=8, range=(0, np.pi), weights=mag); h = h / (h.sum() + 1e-9)
        Jxx = (gx[me] ** 2).mean(); Jyy = (gy[me] ** 2).mean(); Jxy = (gx[me] * gy[me]).mean()
        coh = np.sqrt((Jxx - Jyy) ** 2 + 4 * Jxy ** 2) / (Jxx + Jyy + 1e-9)    # koherensi orientasi (kejelasan striasi)
        dom = 0.5 * np.arctan2(2 * Jxy, Jxx - Jyy) - np.arctan2(np.sin(ph).mean(), np.cos(ph).mean())
        out += [mag.mean(), mag.std(), coh, np.cos(2 * dom), np.sin(2 * dom), *h]
    out += [gn[me].std()]
    return np.nan_to_num(np.array(out, float))

RADIAL_PATH = OUT_DIR + 'emb_radial_feats.npy'
if os.path.exists(RADIAL_PATH):
    RAD = np.load(RADIAL_PATH)
else:
    RAD = np.stack(Parallel(n_jobs=-1)(delayed(radial_feats)(p, f) for p, f in zip(df['path'], df['flip']))).astype(np.float32)
    np.save(RADIAL_PATH, RAD)
print('Fitur radial:', RAD.shape)

In [ ]:
# ================= EKSTRAKSI EMBEDDING (sekali; hasil disimpan dan dipakai ulang) =================
PREPROCESS = {'MobileNetV2': keras.applications.mobilenet_v2.preprocess_input,
              'ResNet50':    keras.applications.resnet50.preprocess_input,
              'InceptionV3': keras.applications.inception_v3.preprocess_input}
BUILDERS = {'MobileNetV2': keras.applications.MobileNetV2, 'ResNet50': keras.applications.ResNet50,
            'InceptionV3': keras.applications.InceptionV3}
# GAP dari beberapa tahap (tengah -> akhir): fitur tengah lebih peka terhadap tekstur serat RNFL
LAYER_NAMES = {'ResNet50':    ['conv3_block4_out', 'conv4_block6_out', 'conv5_block3_out'],
               'MobileNetV2': ['block_6_expand_relu', 'block_13_expand_relu', 'out_relu'],
               'InceptionV3': ['mixed2', 'mixed7', 'mixed10']}

def crop_wedge(img):
    m = tf.reduce_max(img, axis=-1) > 12
    idx = tf.cast(tf.where(m), tf.int32)
    def _crop():
        y0 = tf.reduce_min(idx[:, 0]); x0 = tf.reduce_min(idx[:, 1])
        y1 = tf.reduce_max(idx[:, 0]) + 1; x1 = tf.reduce_max(idx[:, 1]) + 1
        return img[y0:y1, x0:x1, :]
    return tf.cond(tf.shape(idx)[0] > 20, _crop, lambda: img)

def extract_embeddings(arch):
    path = f'{OUT_DIR}emb_{arch}_{EMB_SIZE[0]}.npy'
    if os.path.exists(path):
        E = np.load(path); print(arch, 'embedding ada:', E.shape); return E
    keras.backend.clear_session()
    base = BUILDERS[arch](include_top=False, weights='imagenet', input_shape=EMB_SIZE + (3,))
    outs = []
    for n in LAYER_NAMES[arch]:
        try: outs.append(layers.GlobalAveragePooling2D()(base.get_layer(n).output))
        except ValueError: print('  layer tidak ditemukan, dilewati:', n)
    if not outs: outs = [layers.GlobalAveragePooling2D()(base.output)]
    fe = keras.Model(base.input, outs)
    pre = PREPROCESS[arch]
    def load(p, fl):
        img = tf.io.decode_image(tf.io.read_file(p), channels=3, expand_animations=False)
        img = crop_wedge(tf.cast(img, tf.float32))
        img = tf.cond(fl > 0, lambda: tf.image.flip_left_right(img), lambda: img)   # mata kiri -> orientasi mata kanan
        return pre(tf.image.resize_with_pad(img, EMB_SIZE[0], EMB_SIZE[1]))
    ds = (tf.data.Dataset.from_tensor_slices((df['path'].values, df['flip'].values))
          .map(load, num_parallel_calls=tf.data.AUTOTUNE).batch(64).prefetch(tf.data.AUTOTUNE))
    feats = fe.predict(ds, verbose=0)
    E = np.concatenate(feats, axis=1) if isinstance(feats, (list, tuple)) else feats
    np.save(path, E.astype(np.float32)); print(arch, 'embedding disimpan:', E.shape); return E

for a in BACKBONES: extract_embeddings(a)

In [ ]:
# ================= NESTED CV per pasien: PCA(embedding) + fitur radial + posisi sektor -> Logistic Regression =================
folds = list(StratifiedGroupKFold(N_FOLDS, shuffle=True, random_state=SEED).split(df['path'], df['y'], df['group']))
S = np.eye(12)[df['csec'].values]                       # posisi sektor kanonik (one-hot)
y_all, groups = df['y'].values, df['group'].values
SCORING = 'roc_auc' if NUM_CLASSES == 2 else 'roc_auc_ovr'

def make_pipe(D):
    lr = LogisticRegression(class_weight=CLASS_WEIGHT, max_iter=3000)
    if D == 0:                                          # model 'RADIAL': tanpa embedding
        return Pipeline([('sc2', StandardScaler()), ('lr', lr)]), {'lr__C': PARAM_GRID['lr__C']}
    emb = Pipeline([('sc', StandardScaler()), ('pca', PCA(svd_solver='randomized', random_state=SEED))])
    ct = ColumnTransformer([('emb', emb, slice(0, D))], remainder='passthrough')    # kolom setelah D = radial + one-hot sektor
    return Pipeline([('ct', ct), ('sc2', StandardScaler()), ('lr', lr)]), PARAM_GRID

for arch in ARCHS:
    E = np.zeros((len(df), 0), np.float32) if arch == 'RADIAL' else np.load(f'{OUT_DIR}emb_{arch}_{EMB_SIZE[0]}.npy')
    extra = [RAD, S] if (USE_RADIAL or arch == 'RADIAL') else [S]
    M = np.hstack([E] + extra); D = E.shape[1]
    prob = np.zeros((len(df), NUM_CLASSES)); fold_rows = []
    for k, (tr, va) in enumerate(folds):
        pipe, grid = make_pipe(D)
        gs = GridSearchCV(pipe, grid, scoring=SCORING, n_jobs=-1, refit=True,
                          cv=StratifiedGroupKFold(3, shuffle=True, random_state=SEED))
        gs.fit(M[tr], y_all[tr], groups=groups[tr])              # hyperparameter dipilih HANYA dari data train fold ini
        prob[va] = gs.predict_proba(M[va]); pr = prob[va].argmax(1)
        fold_rows.append(f1_score(y_all[va], pr, average='macro'))
        print(f'{arch} fold {k+1}/{N_FOLDS} | best={gs.best_params_} | inner {SCORING}={gs.best_score_:.3f} | val macroF1={fold_rows[-1]:.3f}')
        pd.DataFrame(prob[va], columns=[f'p{i}' for i in range(NUM_CLASSES)]).assign(
            file_name=df[FILE_COL].values[va], y=y_all[va], fold=k, arch=arch, scenario='emb'
        ).to_csv(f'{OUT_DIR}pred_{RUN_TAG}_emb_{arch}_fold{k}.csv', index=False)
    print(f'>>> {arch}: macroF1 antar-fold = {np.mean(fold_rows):.3f} +- {np.std(fold_rows):.3f}\n')

In [ ]:
# ===== Analisis out-of-fold: ensemble antar-arsitektur, threshold leave-one-fold-out (LOFO), evaluasi TINGKAT MATA =====
pcols = [f'p{i}' for i in range(NUM_CLASSES)]

def load_oof(scenario, arch):
    fs = sorted(glob.glob(f'{OUT_DIR}pred_{RUN_TAG}_{scenario}_{arch}_fold*.csv'))
    return pd.concat([pd.read_csv(f) for f in fs], ignore_index=True) if fs else None

def ensemble(frames):
    allp = pd.concat(frames); g = allp.groupby('file_name')
    out = g[pcols].mean(); out['y'] = g['y'].first(); out['fold'] = g['fold'].first(); out['n_models'] = g.size()
    out = out.reset_index(); return out[out['n_models'] == len(frames)]      # hanya baris yang dinilai SEMUA model

def lofo_pred(score, y, fold, grid):
    """Threshold dipilih (maks macro-F1) pada fold LAIN, lalu diterapkan ke fold ini -> tanpa kebocoran ke fold evaluasi."""
    pred = np.zeros(len(y), int)
    for f in np.unique(fold):
        te = fold == f; tr = ~te
        if tr.sum() == 0 or len(np.unique(y[tr])) < 2: th = grid[len(grid) // 2]
        else: th = grid[int(np.argmax([f1_score(y[tr], (score[tr] >= t).astype(int), average='macro') for t in grid]))]
        pred[te] = (score[te] >= th).astype(int)
    return pred

def bin_report(y, score, fold, grid, fixed_th):
    p0 = (score >= fixed_th).astype(int); pl = lofo_pred(score, y, fold, grid)
    return dict(n=len(y), pos_rate=round(float(np.mean(y)), 3),
                auc=round(roc_auc_score(y, score), 4) if len(np.unique(y)) == 2 else np.nan,
                acc_fixed=round(accuracy_score(y, p0), 4), f1m_fixed=round(f1_score(y, p0, average='macro'), 4),
                acc_lofo=round(accuracy_score(y, pl), 4), balacc_lofo=round(balanced_accuracy_score(y, pl), 4),
                f1m_lofo=round(f1_score(y, pl, average='macro'), 4), f1_pos_lofo=round(f1_score(y, pl, zero_division=0), 4))

def eye_table(d):
    """Agregasi sektor -> mata. Skor = ekspektasi jumlah sektor abnormal (dari p1). Label = >= EYE_ABNORMAL_MIN sektor abnormal (dari label OCT asli)."""
    m = d.merge(df[['file_name', 'unit', 'eye', 'abn']], on='file_name')
    u = m.groupby('unit').agg(eye=('eye', 'first'), fold=('fold', 'first'), n=('p1', 'size'), s=('p1', 'sum'), k=('abn', 'sum')).reset_index()
    u['s'] = u['s'] * 12 / u['n']
    e = u.groupby('eye').agg(fold=('fold', 'first'), score=('s', 'mean'), k=('k', 'max')).reset_index()
    e['y'] = (e['k'] >= EYE_ABNORMAL_MIN).astype(int); e['group'] = e['eye'].str.rsplit('_', n=1).str[0]
    return e

def boot_auc(e, B=500, seed=0):
    rng = np.random.default_rng(seed); groups = e['group'].unique(); by = {g: e[e['group'] == g] for g in groups}; vals = []
    for _ in range(B):
        s = pd.concat([by[g] for g in rng.choice(groups, len(groups))])
        if s['y'].nunique() == 2: vals.append(roc_auc_score(s['y'], s['score']))
    return np.round(np.percentile(vals, [2.5, 97.5]), 3) if vals else (np.nan, np.nan)

def eye_table_stack(d, C=1.0):
    """Stacking tingkat citra: 12 probabilitas sektor kanonik (+ mean, max, jumlah >=0,5) -> LR; dilatih pada fold LAIN (OOF). Lalu rata-rata per mata."""
    m = d.merge(df[['file_name', 'unit', 'eye', 'abn', 'csec']], on='file_name')
    P = m.pivot_table(index='unit', columns='csec', values='p1').reindex(columns=range(12))
    P = P.apply(lambda r: r.fillna(r.mean()), axis=1)
    meta = m.groupby('unit').agg(eye=('eye', 'first'), fold=('fold', 'first'), k=('abn', 'sum')).loc[P.index]
    Z = np.column_stack([P.values, P.mean(1).values, P.max(1).values, (P.values >= 0.5).sum(1)])
    yu = (meta['k'].values >= EYE_ABNORMAL_MIN).astype(int); fu = meta['fold'].values; sc = np.zeros(len(yu))
    for f in np.unique(fu):
        te = fu == f; tr = ~te
        if tr.sum() == 0 or len(np.unique(yu[tr])) < 2: sc[te] = 0.5; continue
        mdl = make_pipeline(StandardScaler(), LogisticRegression(C=C, class_weight='balanced', max_iter=2000)).fit(Z[tr], yu[tr])
        sc[te] = mdl.predict_proba(Z[te])[:, 1]
    u = pd.DataFrame(dict(eye=meta['eye'].values, fold=fu, score=sc, k=meta['k'].values))
    e = u.groupby('eye').agg(fold=('fold', 'first'), score=('score', 'mean'), k=('k', 'max')).reset_index()
    e['y'] = (e['k'] >= EYE_ABNORMAL_MIN).astype(int); e['group'] = e['eye'].str.rsplit('_', n=1).str[0]
    return e

rows_s, rows_e = [], []
for sc, _ in SCENARIOS:
    named = {a: load_oof(sc, a) for a in ARCHS}; named = {a: f for a, f in named.items() if f is not None}
    if len(named) > 1: named['ENSEMBLE(' + '+'.join(named) + ')'] = ensemble(list(named.values()))
    for name, d in named.items():
        y, fold = d['y'].values, d['fold'].values
        if NUM_CLASSES == 2:
            rows_s.append(dict(skenario=sc, model=name, **bin_report(y, d['p1'].values, fold, np.linspace(0.05, 0.95, 91), 0.5)))
            e = eye_table(d)
            r = bin_report(e['y'].values, e['score'].values, e['fold'].values, np.linspace(0, 12, 49), EYE_ABNORMAL_MIN)
            lo, hi = boot_auc(e); r.update(auc_ci95=f'[{lo}, {hi}]')
            rows_e.append(dict(skenario=sc, model=name, **r))
            e2 = eye_table_stack(d)
            r2 = bin_report(e2['y'].values, e2['score'].values, e2['fold'].values, np.linspace(0.05, 0.95, 91), 0.5)
            lo, hi = boot_auc(e2); r2.update(auc_ci95=f'[{lo}, {hi}]')
            rows_e.append(dict(skenario=sc, model=name + ' +STACK', **r2))
        else:
            pr = d[pcols].values.argmax(1)
            rows_s.append(dict(skenario=sc, model=name, n=len(y), acc=round(accuracy_score(y, pr), 4),
                               bal_acc=round(balanced_accuracy_score(y, pr), 4), macro_f1=round(f1_score(y, pr, average='macro'), 4)))
print('=== TINGKAT SEKTOR (out-of-fold, validasi per pasien) ==='); sec_tab = pd.DataFrame(rows_s); display(sec_tab)
sec_tab.to_csv(OUT_DIR + f'analisis_sektor_{RUN_TAG}.csv', index=False)
if rows_e:
    print(f'=== TINGKAT MATA (abnormal = >= {EYE_ABNORMAL_MIN} sektor abnormal; threshold LOFO) ===')
    eye_tab = pd.DataFrame(rows_e); display(eye_tab); eye_tab.to_csv(OUT_DIR + f'analisis_mata_{RUN_TAG}.csv', index=False)
    print('Catatan: label tingkat mata diturunkan dari label sektor (OCT) dengan aturan K sektor; jumlah mata terbatas (~235) -> lihat auc_ci95.')

In [ ]:
# ===== Laporan target (HANYA validasi per pasien, threshold LOFO; bukan angka train) =====
def check(tab, f1col, acccol, level):
    for r in tab.itertuples():
        f1v, av = getattr(r, f1col), getattr(r, acccol)
        ok = (f1v >= TARGET) and (av >= TARGET)
        print(f'[{level}] {r.skenario} | {r.model:40s} acc={av:.3f} F1={f1v:.3f} -> ' + ('TARGET TERCAPAI' if ok else f'di bawah target {TARGET}'))
if NUM_CLASSES == 2:
    check(sec_tab, 'f1m_lofo', 'acc_lofo', 'sektor'); check(eye_tab, 'f1m_lofo', 'acc_lofo', 'mata')
    print('Tingkat mata: lihat juga auc_ci95. Jika batas bawah CI jauh di bawah target, klaim "mencapai target" tidak kuat.')
else:
    check(sec_tab.rename(columns={'macro_f1': 'f1m_lofo', 'acc': 'acc_lofo'}), 'f1m_lofo', 'acc_lofo', 'sektor')

In [ ]:
# ===== CI 95% (bootstrap per pasien) untuk akurasi & macro-F1, threshold LOFO =====
def boot_ci(y, pred, group, B=1000, seed=0):
    """Bootstrap per pasien -> CI 95% untuk akurasi dan macro-F1."""
    idx = pd.Series(np.arange(len(y))).groupby(np.asarray(group)).apply(lambda s: s.values).to_dict()
    keys = list(idx); rng = np.random.default_rng(seed); out = []
    for _ in range(B):
        ii = np.concatenate([idx[k] for k in rng.choice(keys, len(keys))])
        out.append((accuracy_score(y[ii], pred[ii]), f1_score(y[ii], pred[ii], average='macro')))
    lo, hi = np.percentile(out, [2.5, 97.5], axis=0)
    return f'acc [{lo[0]:.3f}, {hi[0]:.3f}] | F1 [{lo[1]:.3f}, {hi[1]:.3f}]'

for sc, _ in SCENARIOS:
    named = {a: load_oof(sc, a) for a in ARCHS}
    named['ENSEMBLE'] = ensemble(list(named.values()))
    for name, d in named.items():
        m = d.merge(df[['file_name', 'group']], on='file_name')
        ps = lofo_pred(m['p1'].values, m['y'].values, m['fold'].values, np.linspace(0.05, 0.95, 91))
        print(f'[sektor] {name:12s}', boot_ci(m['y'].values, ps, m['group'].values))
        e = eye_table(d)
        pe = lofo_pred(e['score'].values, e['y'].values, e['fold'].values, np.linspace(0, 12, 49))
        print(f'[mata]   {name:12s}', boot_ci(e['y'].values, pe, e['group'].values))

## Catatan
- Semua angka berasal dari validasi per pasien (nested CV); hyperparameter dipilih hanya di inner split data train. Target tidak dipakai untuk memilih apa pun.
- Jika target tidak tercapai, laporkan sebagai baseline dengan keterbatasan data (127 pasien, wedge kecil). Itu temuan yang sah.
- Embedding tersimpan (`emb_*.npy`); untuk iterasi lanjut cukup jalankan ulang sel CV tanpa GPU.